In [1]:
!git clone -b feature-data-processing https://github.com/Nirucoder/Swiggy_playground /content/swiggy

Cloning into '/content/swiggy'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 19 (delta 2), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 1.04 MiB | 5.67 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [2]:
!ls /content/swiggy/data/processed/

final_training_data.csv  final_training_data_hourly.csv


In [3]:
import pandas as pd

df = pd.read_csv("/content/swiggy/data/processed/final_training_data.csv")
print(df.shape)
print(df.head())
print(df.columns.tolist())

(16684, 11)
           ds               category             y  temp   rain  \
0  2021-01-03  Continental Beverages  74463.000000  21.5   1.30   
1  2021-01-04  Continental Beverages  73813.142857  21.7   0.20   
2  2021-01-05  Continental Beverages  73163.285714  21.3   0.50   
3  2021-01-06  Continental Beverages  72513.428571  21.5  12.10   
4  2021-01-07  Continental Beverages  71863.571429  22.0  12.95   

   sentiment_score day_of_week    month  is_weekend  sentiment_lag_7  \
0        -0.022968      Sunday  January           1         0.272069   
1         0.316372      Monday  January           0         0.272069   
2         0.434259     Tuesday  January           0         0.272069   
3         0.125000   Wednesday  January           0         0.272069   
4         0.670833    Thursday  January           0         0.272069   

   is_subscriber  
0              1  
1              1  
2              1  
3              1  
4              0  
['ds', 'category', 'y', 'temp', 'rain'

In [4]:
print(df['category'].unique())
print(df['category'].nunique())

['Continental Beverages' 'Continental Pizza' 'Continental Seafood'
 'Indian Beverages' 'Indian Biryani' 'Indian Desert' 'Indian Rice Bowl'
 'Italian Beverages' 'Italian Pasta' 'Italian Sandwich' 'Thai Beverages'
 'Thai Extras' 'Thai Other Snacks' 'Thai Soup' 'Thai Starters'
 'Italian Salad' 'Continental Fish']
17


In [5]:
cols = ['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']
print(df[cols].isnull().sum())

ds                 0
y                  0
rain               0
temp               0
sentiment_lag_7    0
is_subscriber      0
dtype: int64


In [6]:
!pip install prophet -q

In [7]:
from prophet import Prophet

df_base = df[['ds', 'y']].drop_duplicates(subset='ds').dropna()
df_base['ds'] = pd.to_datetime(df_base['ds'])

model_base = Prophet(yearly_seasonality=True, weekly_seasonality=True)
model_base.add_country_holidays(country_name='IN')
model_base.fit(df_base)

print("baseline model trained")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


baseline model trained


In [8]:
df_cat = df[df['category'] == 'Indian Biryani'][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
df_cat['ds'] = pd.to_datetime(df_cat['ds'])

model_full = Prophet(yearly_seasonality=True, weekly_seasonality=True)
model_full.add_country_holidays(country_name='IN')
model_full.add_regressor('rain')
model_full.add_regressor('temp')
model_full.add_regressor('sentiment_lag_7')
model_full.add_regressor('is_subscriber')
model_full.fit(df_cat)

print("full model trained for Indian Biryani")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


full model trained for Indian Biryani


In [9]:
import os
os.makedirs("/content/swiggy/models", exist_ok=True)

In [10]:
import pickle

predicted_totals = {}

for category in df['category'].unique():
    df_cat = df[df['category'] == category][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
    df_cat['ds'] = pd.to_datetime(df_cat['ds'])

    m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
    m.add_country_holidays(country_name='IN')
    m.add_regressor('rain')
    m.add_regressor('temp')
    m.add_regressor('sentiment_lag_7')
    m.add_regressor('is_subscriber')
    m.fit(df_cat)

    future = m.make_future_dataframe(periods=30)
    future['rain'] = df_cat['rain'].mean()
    future['temp'] = df_cat['temp'].mean()
    future['sentiment_lag_7'] = df_cat['sentiment_lag_7'].mean()
    future['is_subscriber'] = df_cat['is_subscriber'].mean()

    forecast = m.predict(future)
    total = forecast['yhat'].tail(30).sum()
    predicted_totals[category] = round(total)

    filename = "/content/swiggy/models/" + category.lower().replace(' ', '_') + ".pkl"
    with open(filename, 'wb') as f:
        pickle.dump(m, f)

    print(f"✅ {category} — predicted orders: {round(total)}")

winner = max(predicted_totals, key=predicted_totals.get)
print(f"\n🏆 Most Selling Product Next Month: {winner} with {predicted_totals[winner]} predicted orders")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Beverages — predicted orders: 1192123


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Pizza — predicted orders: 1209714


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


...

✅ Thai Starters — predicted orders: 365260


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Italian Salad — predicted orders: 1558276


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Fish — predicted orders: 493058

🏆 Most Selling Product Next Month: Indian Rice Bowl with 4575288 predicted orders


In [11]:
!ls /content/swiggy/models/

continental_beverages.pkl  indian_desert.pkl	  thai_beverages.pkl
continental_fish.pkl	   indian_rice_bowl.pkl   thai_extras.pkl
continental_pizza.pkl	   italian_beverages.pkl  thai_other_snacks.pkl
continental_seafood.pkl    italian_pasta.pkl	  thai_soup.pkl
indian_beverages.pkl	   italian_salad.pkl	  thai_starters.pkl
indian_biryani.pkl	   italian_sandwich.pkl


In [12]:
summary_df = pd.DataFrame(
    list(predicted_totals.items()),
    columns=['category', 'predicted_30day_orders']
)
summary_df = summary_df.sort_values('predicted_30day_orders', ascending=False).reset_index(drop=True)
summary_df.to_csv("/content/swiggy/data/processed/forecast_summary.csv", index=False)
print(summary_df)

                 category  predicted_30day_orders
0        Indian Rice Bowl                 4575288
1          Thai Beverages                 3553566
2       Italian Beverages                 2703332
3        Italian Sandwich                 2571605
4           Italian Salad                 1558276
5       Continental Pizza                 1209714
6   Continental Beverages                 1192123
7       Thai Other Snacks                 1085286
8             Thai Extras                  799907
9           Indian Desert                  694487
10    Continental Seafood                  602151
11       Indian Beverages                  497513
12       Continental Fish                  493058
13              Thai Soup                  380920
14          Thai Starters                  365260
15          Italian Pasta                  305429
16         Indian Biryani                  147101


In [13]:
from sklearn.metrics import mean_absolute_error
import numpy as np

df_bt = df[df['category'] == 'Indian Rice Bowl'][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
df_bt['ds'] = pd.to_datetime(df_bt['ds'])
df_bt = df_bt.sort_values('ds').reset_index(drop=True)

cutoff = len(df_bt) - 30
df_train = df_bt.iloc[:cutoff]
df_test = df_bt.iloc[cutoff:]

m_bt = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m_bt.add_country_holidays(country_name='IN')
m_bt.add_regressor('rain')
m_bt.add_regressor('temp')
m_bt.add_regressor('sentiment_lag_7')
m_bt.add_regressor('is_subscriber')
m_bt.fit(df_train)

forecast_bt = m_bt.predict(df_test)

actual = df_test['y'].values
predicted = forecast_bt['yhat'].values

mae = mean_absolute_error(actual, predicted)
mape = np.mean(np.abs((actual - predicted) / actual)) * 100

print(f"MAE  : {mae:.2f} orders")
print(f"MAPE : {mape:.2f}%")
print(f"\nOn average, predictions were off by {mae:.0f} orders per day ({mape:.1f}%)")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


MAE  : 69864.48 orders
MAPE : 62.25%

On average, predictions were off by 69864 orders per day (62.2%)


In [14]:
print(f"Training rows : {len(df_train)}")
print(f"Test rows     : {len(df_test)}")
print(f"\nActual orders range:")
print(df_test['y'].describe())

Training rows : 979
Test rows     : 30

Actual orders range:
count        30.000000
mean     113391.495238
std        8847.495597
min       93183.000000
25%      108029.714286
50%      112010.071429
75%      121139.000000
max      127751.000000
Name: y, dtype: float64


In [15]:
comparison = pd.DataFrame({
    'date': df_test['ds'].values,
    'actual': actual,
    'predicted': predicted.round()
})
print(comparison.to_string())

         date         actual  predicted
0  2023-09-09  112442.857143   178810.0
1  2023-09-10  110136.000000   183342.0
2  2023-09-11  109692.571429   188647.0
3  2023-09-12  109249.142857   192963.0
4  2023-09-13  108805.714286   197616.0
5  2023-09-14  108362.285714   198754.0
6  2023-09-15  107918.857143   196646.0
7  2023-09-16  107475.428571   206366.0
8  2023-09-17  107032.000000   203004.0
9  2023-09-18  109991.857143   203116.0
10 2023-09-19  112951.714286   200240.0
11 2023-09-20  115911.571429   199413.0
12 2023-09-21  118871.428571   198580.0
13 2023-09-22  121831.285714   198851.0
14 2023-09-23  124791.142857   194605.0
15 2023-09-24  127751.000000   193469.0
16 2023-09-25  126491.571429   190057.0
17 2023-09-26  125232.142857   186939.0
18 2023-09-27  123972.714286   184970.0
19 2023-09-28  122713.285714   152487.0
20 2023-09-29  121453.857143   181689.0
21 2023-09-30  120194.428571   176331.0
22 2023-10-01  118935.000000   173565.0
23 2023-10-02  115256.142857   145703.0


In [16]:
from sklearn.preprocessing import MinMaxScaler

df_bt2 = df[df['category'] == 'Indian Rice Bowl'][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
df_bt2['ds'] = pd.to_datetime(df_bt2['ds'])
df_bt2 = df_bt2.sort_values('ds').reset_index(drop=True)

# scale y to 0-1 range
scaler = MinMaxScaler()
df_bt2['y'] = scaler.fit_transform(df_bt2[['y']])

cutoff = len(df_bt2) - 30
df_train2 = df_bt2.iloc[:cutoff]
df_test2 = df_bt2.iloc[cutoff:]

m_bt2 = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.3
)
m_bt2.add_country_holidays(country_name='IN')
m_bt2.add_regressor('rain')
m_bt2.add_regressor('temp')
m_bt2.add_regressor('sentiment_lag_7')
m_bt2.add_regressor('is_subscriber')
m_bt2.fit(df_train2)

forecast_bt2 = m_bt2.predict(df_test2)

actual2 = df_test2['y'].values
predicted2 = forecast_bt2['yhat'].values

mae2 = mean_absolute_error(actual2, predicted2)
mape2 = np.mean(np.abs((actual2 - predicted2) / actual2)) * 100

print(f"Before — MAPE: 62.25%")
print(f"After  — MAPE: {mape2:.2f}%")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Before — MAPE: 62.25%
After  — MAPE: 104.87%


In [17]:
df_bt3 = df[df['category'] == 'Indian Rice Bowl'][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
df_bt3['ds'] = pd.to_datetime(df_bt3['ds'])
df_bt3 = df_bt3.sort_values('ds').reset_index(drop=True)

cutoff = len(df_bt3) - 30
df_train3 = df_bt3.iloc[:cutoff]
df_test3 = df_bt3.iloc[cutoff:]

for cp in [0.1, 0.3, 0.5]:
    m_test = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=cp
    )
    m_test.add_country_holidays(country_name='IN')
    m_test.add_regressor('rain')
    m_test.add_regressor('temp')
    m_test.add_regressor('sentiment_lag_7')
    m_test.add_regressor('is_subscriber')
    m_test.fit(df_train3)

    fc = m_test.predict(df_test3)
    mape_test = np.mean(np.abs((df_test3['y'].values - fc['yhat'].values) / df_test3['y'].values)) * 100
    print(f"changepoint_prior_scale={cp} → MAPE: {mape_test:.2f}%")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=0.1 → MAPE: 62.66%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=0.3 → MAPE: 61.07%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=0.5 → MAPE: 53.72%


In [18]:
for cp in [0.7, 0.9, 1.5]:
    m_test = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=cp
    )
    m_test.add_country_holidays(country_name='IN')
    m_test.add_regressor('rain')
    m_test.add_regressor('temp')
    m_test.add_regressor('sentiment_lag_7')
    m_test.add_regressor('is_subscriber')
    m_test.fit(df_train3)

    fc = m_test.predict(df_test3)
    mape_test = np.mean(np.abs((df_test3['y'].values - fc['yhat'].values) / df_test3['y'].values)) * 100
    print(f"changepoint_prior_scale={cp} → MAPE: {mape_test:.2f}%")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=0.7 → MAPE: 45.95%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=0.9 → MAPE: 41.01%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=1.5 → MAPE: 30.73%


In [19]:
for cp in [2.0, 3.0, 5.0]:
    m_test = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=cp
    )
    m_test.add_country_holidays(country_name='IN')
    m_test.add_regressor('rain')
    m_test.add_regressor('temp')
    m_test.add_regressor('sentiment_lag_7')
    m_test.add_regressor('is_subscriber')
    m_test.fit(df_train3)

    fc = m_test.predict(df_test3)
    mape_test = np.mean(np.abs((df_test3['y'].values - fc['yhat'].values) / df_test3['y'].values)) * 100
    print(f"changepoint_prior_scale={cp} → MAPE: {mape_test:.2f}%")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=2.0 → MAPE: 30.57%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=3.0 → MAPE: 34.25%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


changepoint_prior_scale=5.0 → MAPE: 47.44%


In [20]:
predicted_totals = {}

for category in df['category'].unique():
    df_cat = df[df['category'] == category][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
    df_cat['ds'] = pd.to_datetime(df_cat['ds'])

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=2.0
    )
    m.add_country_holidays(country_name='IN')
    m.add_regressor('rain')
    m.add_regressor('temp')
    m.add_regressor('sentiment_lag_7')
    m.add_regressor('is_subscriber')
    m.fit(df_cat)

    future = m.make_future_dataframe(periods=30)
    future['rain'] = df_cat['rain'].mean()
    future['temp'] = df_cat['temp'].mean()
    future['sentiment_lag_7'] = df_cat['sentiment_lag_7'].mean()
    future['is_subscriber'] = df_cat['is_subscriber'].mean()

    forecast = m.predict(future)
    total = forecast['yhat'].tail(30).sum()
    predicted_totals[category] = round(total)

    filename = "/content/swiggy/models/" + category.lower().replace(' ', '_') + ".pkl"
    with open(filename, 'wb') as f:
        pickle.dump(m, f)

    print(f"✅ {category} — predicted orders: {round(total)}")

winner = max(predicted_totals, key=predicted_totals.get)
print(f"\n🏆 Most Selling Product Next Month: {winner} with {predicted_totals[winner]} predicted orders")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Beverages — predicted orders: 1278751


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Pizza — predicted orders: 481369


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


...

✅ Thai Starters — predicted orders: 772072


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Italian Salad — predicted orders: 2864967


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


✅ Continental Fish — predicted orders: 377905

🏆 Most Selling Product Next Month: Italian Beverages with 3401976 predicted orders


In [21]:
m_final_bt = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=2.0
)
m_final_bt.add_country_holidays(country_name='IN')
m_final_bt.add_regressor('rain')
m_final_bt.add_regressor('temp')
m_final_bt.add_regressor('sentiment_lag_7')
m_final_bt.add_regressor('is_subscriber')
m_final_bt.fit(df_train3)

fc_final = m_final_bt.predict(df_test3)

actual_final = df_test3['y'].values
predicted_final = fc_final['yhat'].values

mae_final = mean_absolute_error(actual_final, predicted_final)
mape_final = np.mean(np.abs((actual_final - predicted_final) / actual_final)) * 100

print(f"Before tuning — MAPE: 62.25%")
print(f"After tuning  — MAPE: {mape_final:.2f}%")
print(f"MAE: {mae_final:.2f} orders per day")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Before tuning — MAPE: 62.25%
After tuning  — MAPE: 30.57%
MAE: 34072.68 orders per day


In [22]:
comparison_final = pd.DataFrame({
    'date': df_test3['ds'].values,
    'actual': actual_final,
    'predicted': predicted_final.round()
})
comparison_final.to_csv("/content/swiggy/data/processed/backtest_results.csv", index=False)
print("saved backtest_results.csv")

saved backtest_results.csv


In [23]:
from prophet.diagnostics import cross_validation, performance_metrics

df_cv_base = df[df['category'] == 'Indian Rice Bowl'][['ds', 'y', 'rain', 'temp', 'sentiment_lag_7', 'is_subscriber']].copy()
df_cv_base['ds'] = pd.to_datetime(df_cv_base['ds'])

m_cv = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=2.0
)
m_cv.add_country_holidays(country_name='IN')
m_cv.add_regressor('rain')
m_cv.add_regressor('temp')
m_cv.add_regressor('sentiment_lag_7')
m_cv.add_regressor('is_subscriber')
m_cv.fit(df_cv_base)

df_cv = cross_validation(
    m_cv,
    initial='500 days',
    period='30 days',
    horizon='30 days'
)

metrics = performance_metrics(df_cv)
print(metrics[['horizon', 'mae', 'mape']].to_string())

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:Making 16 forecasts with cutoffs between 2022-06-15 00:00:00 and 2023-09-08 00:00:00


  0%|          | 0/16 [00:00<?, ?it/s]

   horizon            mae      mape
0   3 days   38337.103334  0.239635
1   4 days   42229.544033  0.267053
2   5 days   46703.989076  0.294453
3   6 days   52954.967321  0.334516
4   7 days   59533.120326  0.376501
5   8 days   66715.386540  0.426141
6   9 days   75085.620286  0.478062
7  10 days   84939.127300  0.539429
8  11 days   94630.737924  0.605587
9  12 days  104686.161044  0.681910
10 13 days  114189.403526  0.756593
11 14 days  123121.356280  0.824622
12 15 days  128730.947987  0.870352
13 16 days  132379.029605  0.913551
14 17 days  135990.192215  0.958956
15 18 days  140897.369212  1.018828
16 19 days  146813.142099  1.081272
17 20 days  155245.757346  1.167687
18 21 days  163816.910538  1.257358
19 22 days  173067.480043  1.360527
20 23 days  179837.272428  1.454743
21 24 days  186201.573590  1.549327
22 25 days  190852.215089  1.601411
23 26 days  192744.366976  1.593610
24 27 days  194994.361501  1.565512
25 28 days  196657.304107  1.542095
26 29 days  198410.279838  1

In [24]:
for sp in [5.0, 10.0, 15.0, 20.0]:
    m_test = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        changepoint_prior_scale=2.0,
        seasonality_prior_scale=sp
    )
    m_test.add_country_holidays(country_name='IN')
    m_test.add_regressor('rain')
    m_test.add_regressor('temp')
    m_test.add_regressor('sentiment_lag_7')
    m_test.add_regressor('is_subscriber')
    m_test.fit(df_train3)

    fc = m_test.predict(df_test3)
    mape_test = np.mean(np.abs((df_test3['y'].values - fc['yhat'].values) / df_test3['y'].values)) * 100
    print(f"seasonality_prior_scale={sp} → MAPE: {mape_test:.2f}%")

/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


seasonality_prior_scale=5.0 → MAPE: 30.53%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


seasonality_prior_scale=10.0 → MAPE: 30.57%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


seasonality_prior_scale=15.0 → MAPE: 30.43%


/usr/local/lib/python3.12/dist-packages/holidays/countries/india.py:200: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


seasonality_prior_scale=20.0 → MAPE: 30.45%


In [25]:
summary_df = pd.DataFrame(
    list(predicted_totals.items()),
    columns=['category', 'predicted_30day_orders']
)
summary_df = summary_df.sort_values('predicted_30day_orders', ascending=False).reset_index(drop=True)
summary_df.to_csv("/content/swiggy/data/processed/forecast_summary.csv", index=False)
print(summary_df)

                 category  predicted_30day_orders
0       Italian Beverages                 3401976
1        Indian Rice Bowl                 3394844
2          Thai Beverages                 3106729
3           Italian Salad                 2864967
4        Italian Sandwich                 2455489
5   Continental Beverages                 1278751
6       Thai Other Snacks                 1138874
7             Thai Extras                  797968
8           Thai Starters                  772072
9           Indian Desert                  612854
10      Continental Pizza                  481369
11       Continental Fish                  377905
12    Continental Seafood                  317942
13              Thai Soup                  296034
14          Italian Pasta                  290597
15       Indian Beverages                  150655
16         Indian Biryani                  116810


In [26]:
!ls /content/swiggy/data/processed/
!echo "---"
!ls /content/swiggy/models/

backtest_results.csv	 final_training_data_hourly.csv
final_training_data.csv  forecast_summary.csv
---
continental_beverages.pkl  indian_desert.pkl	  thai_beverages.pkl
continental_fish.pkl	   indian_rice_bowl.pkl   thai_extras.pkl
continental_pizza.pkl	   italian_beverages.pkl  thai_other_snacks.pkl
continental_seafood.pkl    italian_pasta.pkl	  thai_soup.pkl
indian_beverages.pkl	   italian_salad.pkl	  thai_starters.pkl
indian_biryani.pkl	   italian_sandwich.pkl


In [27]:
df_hourly = pd.read_csv("/content/swiggy/data/processed/final_training_data_hourly.csv")
print(df_hourly.shape)
print(df_hourly.head())
print(df_hourly.columns.tolist())

(73848, 12)
                    ds               category           y  temp  rain  \
0  2023-04-11 00:00:00  Continental Beverages  150.597857  27.4   0.0   
1  2023-04-11 01:00:00  Continental Beverages   60.239143  27.4   0.0   
2  2023-04-11 02:00:00  Continental Beverages   30.119571  27.4   0.0   
3  2023-04-11 03:00:00  Continental Beverages   30.119571  27.4   0.0   
4  2023-04-11 04:00:00  Continental Beverages   30.119571  27.4   0.0   

   sentiment_score day_of_week  month  is_weekend  sentiment_lag_7  \
0          0.15928     Tuesday  April           0         0.048333   
1          0.15928     Tuesday  April           0         0.048333   
2          0.15928     Tuesday  April           0         0.048333   
3          0.15928     Tuesday  April           0         0.048333   
4          0.15928     Tuesday  April           0         0.048333   

   is_subscriber  hour  
0              1     0  
1              1     1  
2              1     2  
3              1     3  
4  

In [28]:
# define peak hours — lunch (12-14) and dinner (19-22)
peak_hours = [12, 13, 14, 19, 20, 21, 22]

df_hourly['ds_date'] = pd.to_datetime(df_hourly['ds']).dt.date

# for each category + date, calculate peak hour ratio
peak_ratio = df_hourly.groupby(['ds_date', 'category']).apply(
    lambda x: x[x['hour'].isin(peak_hours)]['y'].sum() / x['y'].sum()
).reset_index()

peak_ratio.columns = ['ds_date', 'category', 'peak_hour_ratio']

print(peak_ratio.head(10))
print(peak_ratio.shape)

      ds_date               category  peak_hour_ratio
0  2023-04-11  Continental Beverages         0.648649
1  2023-04-11       Continental Fish         0.648649
2  2023-04-11      Continental Pizza         0.648649
3  2023-04-11    Continental Seafood         0.648649
4  2023-04-11       Indian Beverages         0.648649
5  2023-04-11         Indian Biryani         0.648649
6  2023-04-11          Indian Desert         0.648649
7  2023-04-11       Indian Rice Bowl         0.648649
8  2023-04-11      Italian Beverages         0.648649
9  2023-04-11          Italian Pasta         0.648649
(3077, 3)


/tmp/ipykernel_17399/310535532.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  peak_ratio = df_hourly.groupby(['ds_date', 'category']).apply(


In [29]:
# load the hourly distribution pattern
df_hourly['hour'] = pd.to_datetime(df_hourly['ds']).dt.hour

hourly_weights = df_hourly.groupby('hour')['y'].sum()
hourly_weights = hourly_weights / hourly_weights.sum()

print(hourly_weights)

hour
0     0.003861
1     0.001544
2     0.000772
3     0.000772
4     0.000772
5     0.003861
6     0.007722
7     0.015444
8     0.023166
9     0.023166
10    0.030888
11    0.061776
12    0.092664
13    0.115830
14    0.077220
15    0.038610
16    0.030888
17    0.030888
18    0.038610
19    0.077220
20    0.092664
21    0.115830
22    0.077220
23    0.038610
Name: y, dtype: float64


In [30]:
import itertools

# load our forecast summary
forecast_summary = pd.read_csv("/content/swiggy/data/processed/forecast_summary.csv")

# create hourly breakdown
rows = []
for _, row in forecast_summary.iterrows():
    category = row['category']
    total_orders = row['predicted_30day_orders']

    for hour, weight in hourly_weights.items():
        rows.append({
            'category': category,
            'hour': hour,
            'predicted_orders': round(total_orders * weight)
        })

hourly_forecast = pd.DataFrame(rows)
hourly_forecast.to_csv("/content/swiggy/data/processed/hourly_forecast.csv", index=False)
print(hourly_forecast.head(24))
print(f"\nTotal rows: {len(hourly_forecast)}")

             category  hour  predicted_orders
0   Italian Beverages     0             13135
1   Italian Beverages     1              5254
2   Italian Beverages     2              2627
3   Italian Beverages     3              2627
4   Italian Beverages     4              2627
5   Italian Beverages     5             13135
6   Italian Beverages     6             26270
7   Italian Beverages     7             52540
8   Italian Beverages     8             78810
9   Italian Beverages     9             78810
10  Italian Beverages    10            105080
11  Italian Beverages    11            210161
12  Italian Beverages    12            315241
13  Italian Beverages    13            394051
14  Italian Beverages    14            262701
15  Italian Beverages    15            131350
16  Italian Beverages    16            105080
17  Italian Beverages    17            105080
18  Italian Beverages    18            131350
19  Italian Beverages    19            262701
20  Italian Beverages    20       

In [31]:
peak_hours = [12, 13, 14, 19, 20, 21, 22]

peak_by_category = hourly_forecast[hourly_forecast['hour'].isin(peak_hours)].groupby('category')['predicted_orders'].sum().reset_index()
peak_by_category.columns = ['category', 'predicted_peak_orders']

forecast_summary = forecast_summary.merge(peak_by_category, on='category')
forecast_summary.to_csv("/content/swiggy/data/processed/forecast_summary.csv", index=False)
print(forecast_summary)

                 category  predicted_30day_orders  predicted_peak_orders
0       Italian Beverages                 3401976                2206687
1        Indian Rice Bowl                 3394844                2202060
2          Thai Beverages                 3106729                2015176
3           Italian Salad                 2864967                1858357
4        Italian Sandwich                 2455489                1592751
5   Continental Beverages                 1278751                 829459
6       Thai Other Snacks                 1138874                 738730
7             Thai Extras                  797968                 517601
8           Thai Starters                  772072                 500801
9           Indian Desert                  612854                 397529
10      Continental Pizza                  481369                 312239
11       Continental Fish                  377905                 245128
12    Continental Seafood                  317942  